#### StudyBuild | FMCG Inventory Data Integration ETL Pipeline
##### Produces the four fragment raw CSV sources with *engineered* inconsistencies, exactly as the project brief describes (fragmented, dirty raw layer).

##### Output goes to:  data/raw/

In [1]:
from __future__ import annotations

import random
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
PROJECT_DIR = Path.cwd()
RAW_DIR     = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42  # <-- reproducibility: change only if you WANT a new dataset
rng = np.random.default_rng(SEED)
random.seed(SEED)

START = date(2026, 6, 1)  # business window we simulate
END   = date(2026, 8, 31)

SUPPORTED_STATUSES = ["pending", "processing", "shipped", "delivered", "cancelled"]
SKU_POOL = [f"SKU-{i:04d}" for i in range(1, 61)]     # 60 real products (some will be orphaned)
SKU_JUNK = ["", " ", "sku-1001", "UNKNOWN", "NA", None]  # that sometimes leak in as garbage

In [16]:
def _rand_days(start: date, end: date) -> timedelta: 
    """A random number of days inside [start, end]."""
    delta = (end - start).days
    return timedelta(days=int(rng.integers(0, delta + 1)))

In [5]:
def _pick_dirty_date_format() -> str:
    """Return a random str format; several are non-ISO so Pandas readers struggle."""
    fmt = rng.choice(
        [
            "%Y-%m-%d",        # clean ISO (fine)
            "%d/%m/%Y",        # dmy with slashes
            "%m/%d/%Y",        # us std
            "%Y/%m/%d %H:%M",  # with time
            "not-a-date",      # garbage -> cannot parse
        ]
    )
    if fmt == "not-a-date":
        return rng.choice(["not-a-date", "2026-13-45", "31/31/2026", ""])
    return fmt

In [6]:
def _maybe_junk(prob: float) -> bool:
    """With probability `prob`, return True (meaning: inject a defect)."""
    return bool(rng.random() < prob)

In [7]:
def build_warehouses(n_wh: int = 8) -> pd.DataFrame:
    """warehouses.csv — the only *trusted* reference (master) table."""
    cities = ["Tehran", "Isfahan", "Shiraz", "Tabriz", "Mashhad", "Ahvaz", "Karaj", "Rasht"]
    regions = ["North", "South", "Center", "East", "West", "North", "Center", "North"]

    rows = [
        {
            "warehouse_id": f"WH-{i+1:03d}",
            "name": f"Warehouse {random.choice(['Alpha', 'Beta', 'Gamma', 'Delta'])} {i+1}",
            "city": cities[i],
            "region": regions[i],
        }
        for i in range(n_wh)
    ]
    return pd.DataFrame(rows)

In [18]:
def build_sales(valid_wh:set[str], valid_sku:list[str], n_rows:int = 1800) -> pd.DataFrame:
    """Build the *fragmented, dirty* sales source CSV.

    Columns follow the project brief exactly:
        transaction_id, date, sku, warehouse_id, quantity, revenue

    Engineered defects:
      - ~9%  warehouse_id missing / unknown / whitespace-padded
      - ~11% sku blank, case-broken, or NOT in the master SKU list
      - ~13% date in non-ISO format or un-parseable / out-of-range
      - ~7%  quantity <= 0  (returns / data-entry errors)
      - ~8%  revenue absent or negative
      -      a handful of full primary-key duplicates (same txn_id)
    """
    rows: list[dict] = []

    # ----------------------------------------------------------------------
    # PASS 1 — build the main body of rows (clean by construction, then dirty)
    # ----------------------------------------------------------------------
    for i in range(n_rows):
        txn = f"TXN-{i+1:06d}"

        # ---- warehouse ---------------------------------------------------
        if _maybe_junk(0.09):                        # unknown / malformed
            warehouse_id = rng.choice(["WH-999", "WC-001", "WH-0", "", "  WH-005  "])
        else:  
            warehouse_id = rng.choice(list(valid_wh))  # guaranteed in master

        # ---- sku ---------------------------------------------------------
        if _maybe_junk(0.11):                        # broken / unknown product code
            sku = rng.choice(SKU_JUNK + ["SKU-9999", "SKU-0000"])
        else:
            sku = rng.choice(valid_sku)              # guaranteed in SKU pool

        # ---- date --------------------------------------------------------
        if _maybe_junk(0.13):                        # un-parseable or out-of-window
            d = _pick_dirty_date_format()
        else:
            d = (START + _rand_days(START, END)).isoformat()

        # ---- quantity ----------------------------------------------------
        if _maybe_junk(0.07):
            quantity = int(rng.integers(-12, 0))     # returns / negative entry
        else:
            quantity = int(rng.integers(1, 51))

        # ---- revenue -----------------------------------------------------
        if _maybe_junk(0.08):
            revenue = round(float(rng.uniform(-500, 0)), 2) \
                      if rng.random() < 0.5 else None
        else:
            revenue = round(float(rng.uniform(10, 1500)), 2)

        rows.append({
            "transaction_id": txn,
            "date": d,
            "sku": sku,
            "warehouse_id": warehouse_id,
            "quantity": quantity,
            "revenue": revenue,
        })

    # build the working dataframe for PASS 2
    df = pd.DataFrame(rows)

    # ----------------------------------------------------------------------
    # PASS 2 — inject a handful of full primary-key duplicates
    # (same transaction_id, all identical columns → breaks PK uniqueness rule)
    # ----------------------------------------------------------------------
    n_dups = max(1, n_rows // 150)                  # a few rows are legit *retries*
    dup_indices = rng.choice(
        df.index, size=n_dups, replace=False
    )
    dup_rows = df.loc[dup_indices].copy()           # clone chosen clean rows

    df = pd.concat([df, dup_rows], ignore_index=True)
    
    # ----------------------------------------------------------------------
    # PASS 3 — shuffle & return
    # ----------------------------------------------------------------------
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)   # befehlt
    return df

In [20]:
def build_orders(valid_wh: set[str],
                 valid_sku: list[str],
                 n_rows: int = 2200) -> pd.DataFrame:
    """Build the *fragmented, dirty* orders source CSV.

    Columns follow the project brief exactly:
        order_id, date, sku, warehouse_id, ordered_quantity, status

    Engineered defects:
      - ~8%  warehouse_id missing / unknown / whitespace-padded
      - ~12% sku blank, case-broken, or NOT in the master SKU list
      - ~11% date in non-ISO format / un-parseable / out-of-range
      - ~9%  ordered_quantity <= 0
      - ~14% status typo / non-standard (e.g. "shipd", "CANCELLED ", "pend")
      -      a handful of full primary-key duplicates
    """
    standard_statuses = ["Pending", "Shipped", "Delivered", "Cancelled"]
    rows: list[dict] = []

    # ----------------------------------------------------------------------
    # PASS 1 — main body (clean by construction, then inject dirt)
    # ----------------------------------------------------------------------
    for i in range(n_rows):
        # ---- order_id (should be unique PK) ------------------------------
        oid = f"ORD-{i+1:07d}"

        # ---- warehouse ----------------------------------------------------
        if _maybe_junk(0.08):
            warehouse_id = rng.choice(["WH-999", "WC-001", "WH-0", "", "  WH-007  "])
        else:
            warehouse_id = rng.choice(list(valid_wh))

        # ---- sku ----------------------------------------------------------
        if _maybe_junk(0.12):
            sku = rng.choice(SKU_JUNK + ["SKU-9999", "SKU-0000"])
        else:
            sku = rng.choice(valid_sku)

        # ---- date ---------------------------------------------------------
        if _maybe_junk(0.11):
            d = _pick_dirty_date_format()
        else:
            d = (START + _rand_days(START, END)).isoformat()

        # ---- ordered_quantity --------------------------------------------
        if _maybe_junk(0.09):
            qty = int(rng.integers(-8, 0))               # negative data-entry
        else:
            qty = int(rng.integers(1, 100))

        # ---- status -------------------------------------------------------
        if _maybe_junk(0.14):
            # corrupted / non-standard status
            status = rng.choice(["shipd", "CANCELLED ", "pend", "DELIVERD",
                                 " shipped", "canceled", "IN TRANSIT "])
        else:
            status = rng.choice(standard_statuses)

        rows.append({
            "order_id": oid,
            "date": d,
            "sku": sku,
            "warehouse_id": warehouse_id,
            "ordered_quantity": qty,
            "status": status,
        })

    df = pd.DataFrame(rows)

    # ----------------------------------------------------------------------
    # PASS 2 — inject full primary-key duplicates
    # ----------------------------------------------------------------------
    n_dups = max(1, n_rows // 160)
    dup_indices = rng.choice(df.index, size=n_dups, replace=False)
    dup_rows = df.loc[dup_indices].copy()
    df = pd.concat([df, dup_rows], ignore_index=True)

    # ----------------------------------------------------------------------
    # PASS 3 — shuffle & return
    # ----------------------------------------------------------------------
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return df

In [29]:
def build_inventory(valid_wh: set[str],
                    valid_sku: list[str],
                    n_rows: int = 2500) -> pd.DataFrame:
    """Build the *fragmented, dirty* inventory snapshot source CSV.

    Columns follow the project brief exactly:
        snapshot_date, sku, warehouse_id, on_hand_qty

    Engineered defects:
      - ~7%  warehouse_id missing / unknown / whitespace-padded
      - ~10% sku blank, case-broken, or NOT in the master SKU list
      - ~9%  snapshot_date in non-ISO format / un-parseable / out-of-range
      - ~9%  on_hand_qty negative or absurdly impossible (e.g. > 1e6)
    """
    rows: list[dict] = []

    for i in range(n_rows):
        # ---- snapshot date ----------------
        if _maybe_junk(0.09):
            d = _pick_dirty_date_format()
        else:
            d = (START + _rand_days(START, END)).isoformat()

        # ---- warehouse ----------------------------------------------------
        if _maybe_junk(0.07):
            warehouse_id = rng.choice(["WH-999", "WC-001", "WH-0", "", "  WH-012  "])
        else:
            warehouse_id = rng.choice(list(valid_wh))

        # ---- sku ----------------------------------------------------------
        if _maybe_junk(0.10):
            sku = rng.choice(SKU_JUNK + ["SKU-9999", "SKU-0000"])
        else:
            sku = rng.choice(valid_sku)

        # ---- on-hand quantity ---------------------------------------------
        if _maybe_junk(0.09):
            # مقادیر غیرممکن: منفی یا بسیار بزرگ
            on_hand_qty = int(rng.choice([
                int(rng.integers(-500, 0)),      # منفی
                int(rng.integers(2_000_000, 5_000_000)),  # غیرواقعی بزرگ
            ]))
        else:
            on_hand_qty = int(rng.integers(0, 2000))   # مقادیر منطقی

        rows.append({
            "snapshot_date": d,
            "sku": sku,
            "warehouse_id": warehouse_id,
            "on_hand_qty": on_hand_qty,
        })

    df = pd.DataFrame(rows)

    # PASS 2 shuffle
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return df

In [33]:
df_wh = build_warehouses(n_wh=8)

wh_id_col = "warehouse_id" if "warehouse_id" in df_wh.columns else df_wh.columns[0]
valid_wh  = set(df_wh[wh_id_col])        # مجموعه معتبر ID انبارها

df_wh.to_csv(RAW_DIR / "warehouses.csv", index=False)
valid_sku = list(SKU_POOL)

In [34]:
df_sales = build_sales(valid_wh=valid_wh, valid_sku=valid_sku, n_rows=1800)
df_sales.to_csv(RAW_DIR / "sales.csv", index=False)

In [52]:
df_orders = build_orders(
    valid_wh=valid_wh,
    valid_sku=valid_sku,
    n_rows=2200,
)
RAW_DIR.mkdir(parents=True, exist_ok=True)
df_orders.to_csv(RAW_DIR / "orders.csv", index=False)

In [55]:
df_inv = build_inventory(
    valid_wh=valid_wh,
    valid_sku=valid_sku,
    n_rows=2500,
)
RAW_DIR.mkdir(parents=True, exist_ok=True)
df_inv.to_csv(RAW_DIR / "inventory.csv", index=False)

In [47]:
def profile_file(df: pd.DataFrame, name: str,
                 date_col: str = None,
                 qty_cols: list = None,
                 id_col: str = None) -> pd.DataFrame:

    n_rows, n_cols = df.shape
    missing = int(df.isna().sum().sum())
    missing_pct = round(missing / (n_rows * n_cols) * 100, 2)

    miss_cols = df.columns[df.isna().sum() > 0].tolist()
    miss_detail = ", ".join(miss_cols) if miss_cols else "—"

    full_dups = int(df.duplicated().sum())

    invalid_dates = None
    if date_col:
        inv = 0
        for v in df[date_col].dropna():
            try:
                pd.to_datetime(v, format="%Y-%m-%d")
            except (ValueError, TypeError):
                inv += 1
        invalid_dates = inv

    neg_total = 0
    if qty_cols:
        for c in qty_cols:
            numeric = pd.to_numeric(df[c], errors="coerce")
            neg_total += int((numeric < 0).sum())

    type_summary = ", ".join(f"{c}:{t}" for c, t in
                             df.dtypes.astype(str).items())

    row = pd.DataFrame([{
        "file":              name,
        "rows":              n_rows,
        "cols":              n_cols,
        "total_missing":     missing,
        "missing_pct":       missing_pct,
        "missing_in_cols":   miss_detail,
        "full_duplicate_rows": full_dups,
        "invalid_dates":     invalid_dates,
        "total_negative_qty": neg_total,
        "dtypes":            type_summary,
    }])

    return row

In [48]:
sales      = pd.read_csv(RAW_DIR / "sales.csv")  
profile_file(  
    sales, "sales",  
    date_col="date",  
    qty_cols=["quantity", "revenue"],  
    id_col="transaction_id",  
)

,file,rows,cols,total_missing,missing_pct,missing_in_cols,full_duplicate_rows,invalid_dates,total_negative_qty,dtypes
0,sales,1812,6,188,1.73,"date, sku, warehouse_id, revenue",12,246,184,"transaction_id:str, date:str, sku:str, warehou..."


In [49]:
orders     = pd.read_csv(RAW_DIR / "orders.csv")  
profile_file(  
    orders, "orders",  
    date_col="date",  
    qty_cols=["ordered_quantity"],  
    id_col="order_id",  
)

,file,rows,cols,total_missing,missing_pct,missing_in_cols,full_duplicate_rows,invalid_dates,total_negative_qty,dtypes
0,orders,2213,6,141,1.06,"date, sku, warehouse_id",13,249,230,"order_id:str, date:str, sku:str, warehouse_id:..."


In [56]:
inventory  = pd.read_csv(RAW_DIR / "inventory.csv")
profile_file(
    inventory, "inventory",
    date_col="snapshot_date",
    qty_cols=["on_hand_qty"],
)

,file,rows,cols,total_missing,missing_pct,missing_in_cols,full_duplicate_rows,invalid_dates,total_negative_qty,dtypes
0,inventory,2500,4,136,1.36,"snapshot_date, sku, warehouse_id",0,200,113,"snapshot_date:str, sku:str, warehouse_id:str, ..."


In [57]:
warehouses = pd.read_csv(RAW_DIR / "warehouses.csv")  
profile_file(  
    warehouses, "warehouses",  
)

,file,rows,cols,total_missing,missing_pct,missing_in_cols,full_duplicate_rows,invalid_dates,total_negative_qty,dtypes
0,warehouses,8,4,0,0.0,—,0,None,0,"warehouse_id:str, name:str, city:str, region:str"
